In [39]:
using Pkg
Pkg.activate(".")
using Hodge
using Vizagrams
using NetworkLayout
using LinearAlgebra
using AbstractAlgebra
using Combinatorics: powerset
using OrderedCollections
Base.map(f::Function, x::OrderedSet) = [f(i) for i in x]

  Activating project at `~/Documents/GitHub/Julia_Tutorials/TDA`


In [40]:
ℤ_2ℤ, = residue_ring(ZZ, 2)

R = ℤ_2ℤ
⊕(x,y) = R(x) + R(y)
⊙(x,y) = R(x) * R(y)


⊙ (generic function with 1 method)

In [7]:
# Set symmetric diffence = (A∪B)\(A∩B)
Δ(x,y) = symdiff(x,y)
Δ(x,y::Nothing) = x
Δ(x::Nothing,y) = y


"""
indexsimplices(K::SimplicialComplex)

Instead of the original simplices, it returns the same structure,
but with index values instead. For example,
```
K = SimplicialComplex([A, B])
simplicial(K) # [],[A],[B],[A,B]
indexsimplicial(K) # [],[1],[2],[1,2]
```
"""
function indexsimplices(K::SimplicialComplex)
    vert = vertices(K)
    return map(x->map(
                v->findfirst(isequal(v),vert),x
            ),
        simplices(K))
end
function indexsimplices(K::SimplicialComplex,n::Int)
    vert = vertices(K)
    return map(x->map(
                v->findfirst(isequal(v),vert),x
            ),
        simplices(K,n))
end

function compute_adjacency_matrix(vertices::Vector, edges::Vector)
    n = length(vertices)
    vertex_to_index = Dict(v => i for (i, v) in enumerate(vertices))
    adj_matrix = zeros(Int, n, n)
    
    for (u, v) in edges
        i, j = vertex_to_index[u], vertex_to_index[v]
        adj_matrix[i, j] = 1
        adj_matrix[j, i] = 1  # Remove this line if the graph is directed
    end
    
    return adj_matrix
end

function compute_adjacency_matrix(K::SimplicialComplex)
    return compute_adjacency_matrix(vertices(K), simplices(K,1))
end
function diagram(K::SimplicialComplex)
    if dimension(K) < 0
        return NilD()
    end
    pos = shell(compute_adjacency_matrix(K))
    r = mapreduce(x->LinearAlgebra.norm(x-pos[1]),max,pos)/10
    if r ≤ 0.0
        r = 1
    end

    v = mapreduce(x->T(x...)Circle(r=0.1),+, pos)
    
    v = S(:fill=>:grey)mapreduce(x->begin 
            T(x[1]...)*(Circle(r=r)+U(r)S(:fill=>:white)TextMark(text=x[2],fontsize=0.8))
        end
            ,+, zip(pos,vertices(K)),init=NilD())

    e = mapreduce(x->Line(pos[x]),+,indexsimplices(K,1),init=NilD())
    f = S(:fillOpacity=>0.1)mapreduce(x->Polygon(pos[x]),+,indexsimplices(K,2),init=NilD())
    return f+e+v
end
function draw(K::SimplicialComplex)
    return drawsvg(diagram(K))
end

draw (generic function with 1 method)

In [8]:
K = SimplicialComplex([[0], [1], [2], [3], [0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [0, 1, 2]])
draw(K)

<svg height="300" viewBox="-1.24 -1.24 2.48 2.48" width="300.0" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="1.0 0.0 6.123233995736766e-17 1.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polyline points="1.0 0.0 6.123233995736766e-17 1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-1.0 1.2246467991473532e-16 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.16000000000000003) rotate(-0.0, 1.0, -0.08000000000000002)" y="-0.08000000000000002">0</text><circle cy="1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="5.551115123125783e-17" /><text x="5.551115123125783e-17" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 -1.8399999999999999) rotate(-0.0, 5.551115123125783e-17, 0.9199999999999999)" y="0.9199999999999999">1</text><circle cy="1.2246467991473532e-16" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="-1.0" /><text x="-1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.15999999999999978) rotate(-0.0, -1.0, -0.07999999999999989)" y="-0.07999999999999989">2</text><circle cy="-1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="-1.942890293094024e-16" /><text x="-1.8041124150158794e-16" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 2.16) rotate(-0.0, -1.8041124150158794e-16, -1.08)" y="-1.08">3</text></svg>

In [9]:
K = SimplicialComplex([[1],[2],[3]])
vertices(K)
simplices(K)

4-element Vector{Vector{Int64}}:
 []
 [1]
 [2]
 [3]

In [10]:
draw(K)

<svg height="300" viewBox="-0.7078460969082658 -1.073871500692704 1.915692193816531 2.147743001385408" width="266.50635094610965" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688767" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.1385640646055104pt; fill: white; " transform="translate(0 0.1385640646055102) rotate(-0.0, 1.0, -0.0692820323027551)" y="-0.0692820323027551">1</text><circle cy="0.8660254037844387" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.4999999999999998" /><text x="-0.4999999999999998" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 -1.5934867429633672) rotate(-0.0, -0.4999999999999998, 0.7967433714816836)" y="0.7967433714816836">2</text><circle cy="-0.8660254037844385" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.5000000000000004" /><text x="-0.5000000000000004" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 1.8706148721743872) rotate(-0.0, -0.5000000000000004, -0.9353074360871936)" y="-0.9353074360871936">3</text></svg>

In [11]:
betti(K),dimension(K)

([3], 0)

In [12]:
K = SimplicialComplex([[1],[2,3]])

betti(K),dimension(K)

([2, 0], 1)

In [13]:
simplices(K,1)

1-element Vector{Vector{Int64}}:
 [2, 3]

In [14]:
draw(K)

<svg height="300" viewBox="-0.7078460969082658 -1.073871500692704 1.915692193816531 2.147743001385408" width="266.50635094610965" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polyline points="-0.4999999999999998 0.8660254037844387 -0.5000000000000004 -0.8660254037844385 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688767" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.1385640646055104pt; fill: white; " transform="translate(0 0.1385640646055102) rotate(-0.0, 1.0, -0.0692820323027551)" y="-0.0692820323027551">1</text><circle cy="0.8660254037844387" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.4999999999999998" /><text x="-0.4999999999999998" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 -1.5934867429633672) rotate(-0.0, -0.4999999999999998, 0.7967433714816836)" y="0.7967433714816836">2</text><circle cy="-0.8660254037844385" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.5000000000000004" /><text x="-0.5000000000000004" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 1.8706148721743872) rotate(-0.0, -0.5000000000000004, -0.9353074360871936)" y="-0.9353074360871936">3</text></svg>

In [15]:
K = SimplicialComplex([[1,2,3]])

betti(K),dimension(K)

([1, 0, 0], 2)

In [16]:
draw(K)

<svg height="300" viewBox="-0.7078460969082658 -1.073871500692704 1.915692193816531 2.147743001385408" width="266.50635094610965" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="1.0 0.0 -0.4999999999999998 0.8660254037844387 -0.5000000000000004 -0.8660254037844385 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -0.4999999999999998 0.8660254037844387 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -0.5000000000000004 -0.8660254037844385 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.4999999999999998 0.8660254037844387 -0.5000000000000004 -0.8660254037844385 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688767" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.1385640646055104pt; fill: white; " transform="translate(0 0.1385640646055102) rotate(-0.0, 1.0, -0.0692820323027551)" y="-0.0692820323027551">1</text><circle cy="0.8660254037844387" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.4999999999999998" /><text x="-0.4999999999999998" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 -1.5934867429633672) rotate(-0.0, -0.4999999999999998, 0.7967433714816836)" y="0.7967433714816836">2</text><circle cy="-0.8660254037844385" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688779" transform="scale(1,-1)" cx="-0.5000000000000004" /><text x="-0.5000000000000004" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551018pt; fill: white; " transform="translate(0 1.8706148721743872) rotate(-0.0, -0.5000000000000004, -0.9353074360871936)" y="-0.9353074360871936">3</text></svg>

In [17]:
K = SimplicialComplex([[1,2,3],[1,4],[4,5,6],[2,7],[7,8,9]])

draw(K)

<svg height="300" viewBox="-1.1760464815088383 -1.2211616137351378 2.412400342231768 2.4423232274702755" width="296.20193825305205" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="1.0 0.0 0.766044443118978 0.6427876096865393 0.17364817766693041 0.984807753012208 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polygon points="-0.4999999999999998 0.8660254037844387 -0.9396926207859083 0.3420201433256689 -0.9396926207859084 -0.34202014332566866 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polygon points="-0.5000000000000004 -0.8660254037844385 0.17364817766692997 -0.9848077530122081 0.7660444431189778 -0.6427876096865396 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polyline points="1.0 0.0 0.766044443118978 0.6427876096865393 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 0.17364817766693041 0.984807753012208 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -0.4999999999999998 0.8660254037844387 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="0.766044443118978 0.6427876096865393 0.17364817766693041 0.984807753012208 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="0.766044443118978 0.6427876096865393 -0.5000000000000004 -0.8660254037844385 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.4999999999999998 0.8660254037844387 -0.9396926207859083 0.3420201433256689 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.4999999999999998 0.8660254037844387 -0.9396926207859084 -0.34202014332566866 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.9396926207859083 0.3420201433256689 -0.9396926207859084 -0.34202014332566866 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.5000000000000004 -0.8660254037844385 0.17364817766692997 -0.9848077530122081 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-0.5000000000000004 -0.8660254037844385 0.7660444431189778 -0.6427876096865396 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="0.17364817766692997 -0.9848077530122081 0.7660444431189778 -0.6427876096865396 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19696155060244158" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.1575692404819531pt; fill: white; " transform="translate(0 0.1575692404819533) rotate(-0.0, 1.0, -0.07878462024097665)" y="-0.07878462024097665">1</text><circle cy="0.6427876096865393" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19696155060244158" transform="scale(1,-1)" cx="0.766044443118978" /><text x="0.766044443118978" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.1575692404819533pt; fill: white; " transform="translate(0 -1.1280059788911252) rotate(-0.0, 0.766044443118978, 0.5640029894455626)" y="0.5640029894455626">2</text><circle cy="0.984807753012208" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19696155060244158" transform="scale(1,-1)" cx="0.17364817766693041" /><text x="0.17364817766693041" style="font-family: Helvetica; text-anchor:

## Orientation and Faces

In [18]:
K = SimplicialComplex([[1,0,3]])
σ = simplices(K)[end]

face(σ,i::Int) = σ[begin:end .!= i]

faces(σ) = map(i->σ[begin:end .!= i[1]],enumerate(σ))

faces(σ)
faces([1,2,3])

3-element Vector{Vector{Int64}}:
 [2, 3]
 [1, 3]
 [1, 2]

## Skeletons, Chains, Cycles, Boundaries

In [19]:
K = SimplicialComplex([[1,2,3]])

@show simplices(skeleton(K,0)) # K₀
@show simplices(skeleton(K,1)) # K₁
@show simplices(skeleton(K,2));# K₂

simplices(skeleton(K, 0)) = [Int64[], [1], [2], [3]]
simplices(skeleton(K, 1)) = [Int64[], [1], [2], [3], [1, 2], [1, 3], [2, 3]]
simplices(skeleton(K, 2)) = [Int64[], [1], [2], [3], [1, 2], [1, 3], [2, 3], [1, 2, 3]]


In [20]:
@show simplices(K,0);# K₍₀₎
@show simplices(K,1);# K₍₁₎
@show simplices(K,2);# K₍₂₎;

simplices(K, 0) = [[1], [2], [3]]
simplices(K, 1) = [[1, 2], [1, 3], [2, 3]]
simplices(K, 2) = [[1, 2, 3]]


In [25]:
K = SimplicialComplex([[0,2],[0,1]])

chains(K::SimplicialComplex, n::Int) = collect(powerset(simplices(K,n)))

chains (generic function with 1 method)

In [26]:
chains(K,0)

8-element Vector{Vector{Vector{Int64}}}:
 []
 [[0]]
 [[2]]
 [[1]]
 [[0], [2]]
 [[0], [1]]
 [[2], [1]]
 [[0], [2], [1]]

In [27]:
@show chains(K,0)
@show chains(K,1);

chains(K, 0) = [Vector{Int64}[], [[0]], [[2]], [[1]], [[0], [2]], [[0], [1]], [[2], [1]], [[0], [2], [1]]]
chains(K, 1) = [Vector{Int64}[], [[0, 2]], [[0, 1]], [[0, 2], [0, 1]]]


In [28]:
c1 = chains(K,0)[6]
c2 = chains(K,0)[5]

@show c1
@show c2
@show Δ(c1, c2);

c1 = [[0], [1]]
c2 = [[0], [2]]
Δ(c1, c2) = [[1], [2]]


#### Computing the Boundary

In [29]:
# Boundary 
∂(σ,n::Int) = filter(x->length(x)==n, collect(powerset(σ)))
∂(σ::Union{OrderedSet,Vector{<:Vector}},n::Int) = mapreduce(x->∂(x,n),Δ, σ,init=[])

ncycles(K::SimplicialComplex, n::Int) = mapreduce(x->(∂(x,n) == [] || ∂(x,n) == [[]]) ? [x] : [],vcat,chains(K,n))
nboundaries(K::SimplicialComplex, n::Int) = map(x->∂(x,n+1),chains(K,n+1))

nboundaries (generic function with 1 method)

In [30]:
K = SimplicialComplex([[1]])
c = chains(K,0)[2]

map(x->∂(x,0), chains(K,0))

2-element Vector{Vector{Any}}:
 []
 [Int64[]]

In [31]:
K = SimplicialComplex([[1], [0], [2], [3], [0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [0, 1, 2]])
draw(K)

<svg height="300" viewBox="-1.24 -1.24 2.48 2.48" width="300.0" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="6.123233995736766e-17 1.0 1.0 0.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 1.0 0.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-1.0 1.2246467991473532e-16 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.16000000000000003) rotate(-0.0, 1.0, -0.08000000000000002)" y="-0.08000000000000002">1</text><circle cy="1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="5.551115123125783e-17" /><text x="5.551115123125783e-17" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 -1.8399999999999999) rotate(-0.0, 5.551115123125783e-17, 0.9199999999999999)" y="0.9199999999999999">0</text><circle cy="1.2246467991473532e-16" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="-1.0" /><text x="-1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.15999999999999978) rotate(-0.0, -1.0, -0.07999999999999989)" y="-0.07999999999999989">2</text><circle cy="-1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="-1.942890293094024e-16" /><text x="-1.8041124150158794e-16" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 2.16) rotate(-0.0, -1.8041124150158794e-16, -1.08)" y="-1.08">3</text></svg>

In [32]:
@show ∂([0,1],1);
@show ∂([[0,1],[1,2],[0,2]],1);
@show ∂([0,1,2],2);
@show ∂(∂([0,1,2],2),1);


@show ncycles(K,1);
@show nboundaries(K,1);

∂([0, 1], 1) = [[0], [1]]
∂([[0, 1], [1, 2], [0, 2]], 1) = Any[]
∂([0, 1, 2], 2) = [[0, 1], [0, 2], [1, 2]]
∂(∂([0, 1, 2], 2), 1) = Any[]
ncycles(K, 1) = Any[Vector{Int64}[], [[1, 2], [1, 3], [2, 3]], [[1, 2], [0, 1], [0, 2]], [[1, 3], [0, 1], [0, 2], [2, 3]]]
nboundaries(K, 1) = Vector{Any}[[], [[0, 1], [0, 2], [1, 2]]]


In [33]:
K = SimplicialComplex([[1,2,3],[3,4]])

∂([1,2],1)

# faces([1,2,3])

2-element Vector{Vector{Int64}}:
 [1]
 [2]

In [34]:
chains(K,1)

16-element Vector{Vector{Vector{Int64}}}:
 []
 [[1, 2]]
 [[1, 3]]
 [[2, 3]]
 [[3, 4]]
 [[1, 2], [1, 3]]
 [[1, 2], [2, 3]]
 [[1, 2], [3, 4]]
 [[1, 3], [2, 3]]
 [[1, 3], [3, 4]]
 [[2, 3], [3, 4]]
 [[1, 2], [1, 3], [2, 3]]
 [[1, 2], [1, 3], [3, 4]]
 [[1, 2], [2, 3], [3, 4]]
 [[1, 3], [2, 3], [3, 4]]
 [[1, 2], [1, 3], [2, 3], [3, 4]]

In [35]:
map(x->∂(x,1),chains(K,1))

16-element Vector{Vector{Any}}:
 []
 [[1], [2]]
 [[1], [3]]
 [[2], [3]]
 [[3], [4]]
 [[2], [3]]
 [[1], [3]]
 [[1], [2], [3], [4]]
 [[1], [2]]
 [[1], [4]]
 [[2], [4]]
 []
 [[2], [4]]
 [[1], [4]]
 [[1], [2], [3], [4]]
 [[3], [4]]

## Incremental Algorithm

In [36]:
"""
k_ordering(K::SimplicialComplex)

Creates an incremental list of simplicial complexes.
"""
function k_ordering(K::SimplicialComplex)
    Ks = []
    Ki = SimplicialComplex()
    for d in 0:dimension(K)
        for s in simplices(K,d)
            insert!(Ki,s)
            push!(Ks,copy(Ki))
        end
    end
    return Ks
end

K = SimplicialComplex([[1], [0], [2], [3], [0, 1], [0, 2], [1, 2], [1, 3], [2, 3], [0, 1, 2]])
draw(K)

<svg height="300" viewBox="-1.24 -1.24 2.48 2.48" width="300.0" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><polygon points="6.123233995736766e-17 1.0 1.0 0.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; fill-opacity: 0.1; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="1.0 0.0 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 1.0 0.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="6.123233995736766e-17 1.0 -1.0 1.2246467991473532e-16 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><polyline points="-1.0 1.2246467991473532e-16 -1.8369701987210297e-16 -1.0 " style="vector-effect: non-scaling-stroke; stroke: black; fill: none; " transform="scale(1,-1)" /><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="1.0" /><text x="1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.16000000000000003) rotate(-0.0, 1.0, -0.08000000000000002)" y="-0.08000000000000002">1</text><circle cy="1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="5.551115123125783e-17" /><text x="5.551115123125783e-17" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 -1.8399999999999999) rotate(-0.0, 5.551115123125783e-17, 0.9199999999999999)" y="0.9199999999999999">0</text><circle cy="1.2246467991473532e-16" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.19999999999999996" transform="scale(1,-1)" cx="-1.0" /><text x="-1.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.15999999999999978) rotate(-0.0, -1.0, -0.07999999999999989)" y="-0.07999999999999989">2</text><circle cy="-1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.2" transform="scale(1,-1)" cx="-1.942890293094024e-16" /><text x="-1.8041124150158794e-16" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000003pt; fill: white; " transform="translate(0 2.16) rotate(-0.0, -1.8041124150158794e-16, -1.08)" y="-1.08">3</text></svg>

In [37]:
drawsvg(mapreduce(diagram, (x,y)-> x→(T(1,0),y), k_ordering(K)))

<svg height="300" viewBox="-1.04 -1.2399999999999993 32.12641016151376 2.4799999999999986" width="4005.8012701892226" preserveAspectRatio="xMidYMid meet" xmlns="http://www.w3.org/2000/svg" version="1.1"><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="1.0" transform="scale(1,-1)" cx="0.0" /><text x="0.0" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.8pt; fill: white; " transform="translate(0 0.8) rotate(-0.0, 0.0, -0.4)" y="-0.4">1</text><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.20000000000000018" transform="scale(1,-1)" cx="4.2" /><text x="4.2" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.16000000000000003) rotate(-0.0, 4.2, -0.08000000000000002)" y="-0.08000000000000002">1</text><circle cy="1.2246467991473532e-16" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.20000000000000018" transform="scale(1,-1)" cx="2.2" /><text x="2.2" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.15999999999999978) rotate(-0.0, 2.2, -0.07999999999999989)" y="-0.07999999999999989">0</text><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688767" transform="scale(1,-1)" cx="7.073205080756889" /><text x="7.073205080756889" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551085pt; fill: white; " transform="translate(0 0.1385640646055102) rotate(-0.0, 7.073205080756889, -0.0692820323027551)" y="-0.0692820323027551">1</text><circle cy="0.8660254037844387" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688767" transform="scale(1,-1)" cx="5.573205080756889" /><text x="5.573205080756889" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551085pt; fill: white; " transform="translate(0 -1.5934867429633672) rotate(-0.0, 5.573205080756889, 0.7967433714816836)" y="0.7967433714816836">0</text><circle cy="-0.8660254037844385" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.17320508075688856" transform="scale(1,-1)" cx="5.573205080756889" /><text x="5.573205080756889" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.13856406460551085pt; fill: white; " transform="translate(0 1.8706148721743872) rotate(-0.0, 5.573205080756889, -0.9353074360871936)" y="-0.9353074360871936">2</text><circle cy="0.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.1999999999999993" transform="scale(1,-1)" cx="10.446410161513777" /><text x="10.446410161513777" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.16000000000000003) rotate(-0.0, 10.446410161513777, -0.08000000000000002)" y="-0.08000000000000002">1</text><circle cy="1.0" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.1999999999999993" transform="scale(1,-1)" cx="9.446410161513777" /><text x="9.446410161513777" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 -1.8399999999999999) rotate(-0.0, 9.446410161513777, 0.9199999999999999)" y="0.9199999999999999">0</text><circle cy="1.2246467991473532e-16" style="vector-effect: non-scaling-stroke; fill: grey; " r="0.1999999999999993" transform="scale(1,-1)" cx="8.446410161513777" /><text x="8.446410161513777" style="font-family: Helvetica; text-anchor: middle; vector-effect: non-scaling-stroke; font-size: 0.16000000000000014pt; fill: white; " transform="translate(0 0.15999999999999978) rotate(-0.0, 8.446410161513777, -0.07999999999999989)" y="

###  Boundary Matrix

In [38]:
K = SimplicialComplex([[1,2,3]])

map(x->∂(x,2),simplices(K))

8-element Vector{Vector{Vector{Int64}}}:
 []
 []
 []
 []
 [[1, 2]]
 [[1, 3]]
 [[2, 3]]
 [[1, 2], [1, 3], [2, 3]]